# Simulating some NR events

Pueh Leng Tan, 10 March 2026

In [16]:
import os
from time import time

import numpy as np
import pandas as pd
import scipy.stats as sps
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import jax.numpy as jnp
import multihist as mh

import appletree as apt
from appletree.utils import get_file_path

In [ ]:
# constrain the GPU memory usage
apt.set_gpu_memory_usage(0.5)

## Define component

### ComponentSim

In [4]:
# The components is associated with bins, so first we load bins
data = pd.read_csv(get_file_path("data_Rn220.csv"))
bins_cs1, bins_cs2 = apt.utils.get_equiprob_bins_2d(
    data[["cs1", "cs2"]].to_numpy(),
    [15, 15],
    order=[0, 1],
    x_clip=[0, 100],
    y_clip=[1e2, 1e4],
    which_np=jnp,
)

In [6]:
# Initialize component
#nr = apt.NR(bins=[bins_cs1, bins_cs2], bins_type="irreg")
nr = apt.NR()

In [7]:
# Deduce the workflow(datastructure)
nr.deduce(data_names=["cs1", "cs2"], func_name="simulate")  # 'eff'(efficiency) is always simulated
nr.rate_name = "nr_rate"  # also we have to specify a normalization factor of the component

# Compile ER script
# This is meta-programing because  appletree can generate codes dynamically
nr.compile()

NR_llh's map s2_cut_acc is using the parameter s2_cut_acc_sigma.
NR_llh's map s1_cut_acc is using the parameter s1_cut_acc_sigma.
NR_llh's map s1_eff_3f is static and not using any parameter.


In [8]:
# For reference, this is the compiled code, the function is stored in appletree.share._cached_functions
# Initialize component
print('NR')
print(nr.code)


NR
from functools import partial
from jax import jit
from appletree.plugins import PositionSpectra
from appletree.plugins import FixedEnergySpectra
from appletree.plugins import ChargeYield
from appletree.plugins import DriftLoss
from appletree.plugins import NumberElectron
from appletree.plugins import S2LCE
from appletree.plugins import ElectronDrifted
from appletree.plugins import S2PE
from appletree.plugins import S2
from appletree.plugins import LightYield
from appletree.plugins import S1LCE
from appletree.plugins import NumberPhoton
from appletree.plugins import PhotonDetection
from appletree.plugins import S1PE
from appletree.plugins import S1
from appletree.plugins import S2CutAccept
from appletree.plugins import S1CutAccept
from appletree.plugins import S1ReconEff
from appletree.plugins import S2Threshold
from appletree.plugins import PositionRecon
from appletree.plugins import S2Correction
from appletree.plugins import S1Correction
from appletree.plugins import Eff
from apple

## Load parameters

In [ ]:
# Of course we have to load parameters (and their priors) in simulation (who the hell writes such comments..)
par_manager = apt.Parameter(get_file_path("nr_low.json"))

({'gas_gain': 31.3,
  'drift_velocity': 0.0677,
  's2_threshold': 500.0,
  'g1': np.float64(0.1522711640923953),
  'g2': np.float64(16.01951586311169),
  'p_dpe': np.float64(0.2409975790385345),
  'elife_sigma': np.float64(-0.024076291205529465),
  's1_eff_3f_sigma': np.float64(-1.4873697014354859),
  's1_cut_acc_sigma': np.float64(0.3670639276322338),
  's2_cut_acc_sigma': np.float64(-0.6380526087827068),
  't_ly': np.float64(0.08483754348448042),
  't_qy': np.float64(-0.02229560316640875),
  'nr_rate': np.float64(1993.7482354797432)},
 13)

## Simulation

In [28]:
num_sigmas = 6
tail_prob = sps.norm.sf(x=num_sigmas)
suggested_max_batch = sps.norm.ppf(1-tail_prob,
                                   loc=par_manager.par_config['nr_rate']['init_mean'],
                                   scale=par_manager.par_config['nr_rate']['init_mean'])
print(suggested_max_batch) # number of NR events hardly gonna fluctuate above this

13999.999981768318


In [140]:
batch_size = int(14e3)
num_sims = int(3000)

param_bag = []
events_bag = []

for _mc in range(num_sims):
    #print(_mc)
    key = apt.randgen.get_key(seed=_mc)
    par_manager.sample_prior() # sampling from prior
    parameters = par_manager.get_all_parameter()

    key, tmp = nr.simulate(key, batch_size, parameters)
    events = np.asarray(jnp.stack(tmp, axis=1))   # shape (n, 3)
    #print(events)
    param_bag.append(parameters.copy())
    events_bag.append(events)

In [144]:
target = 200_000

24.6/3000*target/60. # mins

27.333333333333336

In [146]:
len(events_bag)

3000

In [147]:
# todo:
# 1. save the simulated data
# 2. visualize the simulated data
# 3. think of a way to sample number of events according to the rate parameter